In [1]:
import pandas as pd
import numpy as np
import pickle

# Load data
df = pd.read_csv(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\clean_ratings.csv')

# Load ALS model + encoders
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\als_model.pkl', 'rb') as f:
    als_model = pickle.load(f)

with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)
    user_encoder = encoders['user_encoder']
    user_decoder = encoders['user_decoder']
    product_encoder = encoders['product_encoder']
    product_decoder = encoders['product_decoder']

with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\interaction_matrix.pkl', 'rb') as f:
    interaction_matrix = pickle.load(f)

# Load content-based model
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\combined_matrix.pkl', 'rb') as f:
    combined_matrix = pickle.load(f)

with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\product_profiles.pkl', 'rb') as f:
    product_profiles = pickle.load(f)

with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\product_idx_map.pkl', 'rb') as f:
    idx_maps = pickle.load(f)
    product_idx_map = idx_maps['product_idx_map']
    product_id_map = idx_maps['product_id_map']

print("✅ All models loaded successfully")
print(f"Users: {len(user_encoder):,} | Products: {len(product_encoder):,}")

✅ All models loaded successfully
Users: 10,327 | Products: 2,717


In [2]:
from scipy.sparse import csr_matrix as csr

def get_als_recommendations(user_id, n=20):
    if user_id not in user_encoder:
        return {}

    user_idx = user_encoder[user_id]

    # implicit 0.7.x fix — needs a proper CSR matrix, not a slice
    user_items = csr(interaction_matrix.T[user_idx])

    recommended_ids, scores = als_model.recommend(
        user_idx,
        user_items,
        N=n,
        filter_already_liked_items=True
    )

    results = {}
    max_score = float(max(scores)) if len(scores) > 0 else 1.0

    for pid, score in zip(recommended_ids, scores):
        pid_int = int(pid)
        if pid_int in product_decoder:
            results[product_decoder[pid_int]] = float(score / max_score)

    return results

# Quick test
sample_user = df['user_id'].iloc[0]
als_recs = get_als_recommendations(sample_user)
print(f"ALS returned {len(als_recs)} recommendations")
print("\nSample output:")
for pid, score in list(als_recs.items())[:3]:
    print(f"  {pid} → {score:.4f}")

ALS returned 16 recommendations

Sample output:
  B00001W0DH → 1.0000
  B0002F75LA → 0.9921
  B00004Y2V2 → 0.9545


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

def get_content_recommendations(user_id, n=20):
    user_ratings = df[(df['user_id'] == user_id) & (df['rating'] >= 4)]

    if user_ratings.empty:
        return {}

    score_accumulator = np.zeros(len(product_profiles))

    for _, row in user_ratings.iterrows():
        pid = row['product_id']
        if pid not in product_idx_map:
            continue
        idx = product_idx_map[pid]
        sims = cosine_similarity(
            combined_matrix[idx], combined_matrix
        ).flatten()
        score_accumulator += sims * row['rating']

    # Zero out already rated products
    rated = set(df[df['user_id'] == user_id]['product_id'])
    for pid in rated:
        if pid in product_idx_map:
            score_accumulator[product_idx_map[pid]] = 0

    # Normalize scores to 0-1
    max_score = score_accumulator.max()
    if max_score > 0:
        score_accumulator = score_accumulator / max_score

    top_indices = score_accumulator.argsort()[::-1][:n]
    return {
        product_id_map[idx]: float(score_accumulator[idx])
        for idx in top_indices
        if score_accumulator[idx] > 0
    }

content_recs = get_content_recommendations(sample_user)
print(f"Content-based returned {len(content_recs)} recommendations")

Content-based returned 20 recommendations


In [4]:
def hybrid_recommend(user_id, n=10, als_weight=None, content_weight=None):
    """
    Hybrid recommendation combining ALS + Content-Based filtering.

    Weights are automatically determined by user activity:
    - New users (< 5 reviews)    → 20% ALS, 80% Content
    - Regular users (5-20)       → 50% ALS, 50% Content
    - Power users (20+)          → 80% ALS, 20% Content
    """
    # Auto-determine weights based on user activity
    if als_weight is None or content_weight is None:
        user_review_count = len(df[df['user_id'] == user_id])

        if user_review_count < 5:
            als_weight, content_weight = 0.2, 0.8
            strategy = "content-heavy (new user)"
        elif user_review_count < 20:
            als_weight, content_weight = 0.5, 0.5
            strategy = "balanced"
        else:
            als_weight, content_weight = 0.8, 0.2
            strategy = "ALS-heavy (power user)"
    else:
        strategy = "manual"

    print(f"Strategy: {strategy} | ALS: {als_weight} | Content: {content_weight}")

    # Get recommendations from both models
    als_recs = get_als_recommendations(user_id, n=50)
    content_recs = get_content_recommendations(user_id, n=50)

    # Combine scores
    all_products = set(als_recs.keys()) | set(content_recs.keys())
    hybrid_scores = {}

    for product_id in all_products:
        als_score = als_recs.get(product_id, 0)
        content_score = content_recs.get(product_id, 0)
        hybrid_scores[product_id] = (
            als_weight * als_score +
            content_weight * content_score
        )

    # Sort and return top N
    sorted_recs = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:n]

    results = []
    for product_id, score in sorted_recs:
        profile = product_profiles[
            product_profiles['product_id'] == product_id
        ]
        avg_rating = float(profile['avg_rating'].values[0]) if len(profile) > 0 else 0
        review_count = int(profile['review_count'].values[0]) if len(profile) > 0 else 0

        results.append({
            'product_id': product_id,
            'hybrid_score': round(score, 4),
            'avg_rating': round(avg_rating, 2),
            'review_count': review_count
        })

    return results

In [5]:
print("=" * 55)
print("HYBRID RECOMMENDATION SYSTEM — TEST")
print("=" * 55)

# Test with sample user
sample_user = df['user_id'].iloc[0]
user_review_count = len(df[df['user_id'] == sample_user])
print(f"\nUser: {sample_user}")
print(f"Their review history: {user_review_count} reviews")
print(f"\nTop 10 Recommendations:\n")

recs = hybrid_recommend(sample_user, n=10)

for i, r in enumerate(recs, 1):
    print(f"{i:2}. Product: {r['product_id']}")
    print(f"     Score: {r['hybrid_score']} | "
          f"Avg Rating: {r['avg_rating']}⭐ | "
          f"Reviews: {r['review_count']}")
    print()

HYBRID RECOMMENDATION SYSTEM — TEST

User: A3FO5AKVTFRCRJ
Their review history: 5 reviews

Top 10 Recommendations:

Strategy: balanced | ALS: 0.5 | Content: 0.5
 1. Product: B00001W0DH
     Score: 0.5 | Avg Rating: 4.6⭐ | Reviews: 5

 2. Product: B0002H0A3S
     Score: 0.5 | Avg Rating: 4.71⭐ | Reviews: 809

 3. Product: B0007Y09VO
     Score: 0.4999 | Avg Rating: 4.71⭐ | Reviews: 809

 4. Product: B0002F75LA
     Score: 0.496 | Avg Rating: 4.05⭐ | Reviews: 20

 5. Product: B0002E1NNC
     Score: 0.4808 | Avg Rating: 4.7⭐ | Reviews: 838

 6. Product: B0002E1NWI
     Score: 0.4808 | Avg Rating: 4.7⭐ | Reviews: 838

 7. Product: B00004Y2V2
     Score: 0.4773 | Avg Rating: 5.0⭐ | Reviews: 8

 8. Product: B000068NYM
     Score: 0.4761 | Avg Rating: 4.67⭐ | Reviews: 9

 9. Product: B00000J50W
     Score: 0.4744 | Avg Rating: 4.67⭐ | Reviews: 6

10. Product: B0002F59ZY
     Score: 0.4714 | Avg Rating: 4.86⭐ | Reviews: 7



In [6]:
print("=" * 55)
print("MODEL COMPARISON FOR SAME USER")
print("=" * 55)

als_only = get_als_recommendations(sample_user, n=10)
content_only = get_content_recommendations(sample_user, n=10)
hybrid = hybrid_recommend(sample_user, n=10)

als_products = set(als_only.keys())
content_products = set(content_only.keys())
hybrid_products = set(r['product_id'] for r in hybrid)

print(f"\nALS only products:          {als_products}")
print(f"\nContent-based products:     {content_products}")
print(f"\nHybrid products:            {hybrid_products}")

# How much do they overlap?
als_content_overlap = len(als_products & content_products)
print(f"\nALS ∩ Content overlap:      {als_content_overlap} products")
print(f"Unique to hybrid:           {len(hybrid_products - als_products - content_products)} products")

MODEL COMPARISON FOR SAME USER
Strategy: balanced | ALS: 0.5 | Content: 0.5

ALS only products:          {'B00004UE29', 'B000068NYM', 'B000068NSX', 'B0002F75LA', 'B000068NVJ', 'B00004Y2V2', 'B0002F59ZY', 'B00001W0DH', 'B00000J50W', 'B00001W0DT'}

Content-based products:     {'B0002E1NWI', 'B0002DV6TO', 'B0002H0A3S', 'B0002H04NE', 'B0002H0JZ2', 'B0002DUS8E', 'B0002E1J3Q', 'B0002E1NNC', 'B0006IQLF4', 'B0007Y09VO'}

Hybrid products:            {'B0002E1NWI', 'B000068NYM', 'B0002H0A3S', 'B0002F75LA', 'B00004Y2V2', 'B0002F59ZY', 'B00001W0DH', 'B0002E1NNC', 'B00000J50W', 'B0007Y09VO'}

ALS ∩ Content overlap:      0 products
Unique to hybrid:           0 products


In [7]:
import os

os.makedirs('../src', exist_ok=True)

hybrid_code = '''
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

class HybridRecommender:
    def __init__(self, models_path="../models", data_path="../data"):
        self.df = pd.read_csv(f"{data_path}/clean_ratings.csv")

        with open(f"{models_path}/als_model.pkl", "rb") as f:
            self.als_model = pickle.load(f)
        with open(f"{models_path}/encoders.pkl", "rb") as f:
            enc = pickle.load(f)
            self.user_encoder = enc["user_encoder"]
            self.user_decoder = enc["user_decoder"]
            self.product_encoder = enc["product_encoder"]
            self.product_decoder = enc["product_decoder"]
        with open(f"{models_path}/interaction_matrix.pkl", "rb") as f:
            self.interaction_matrix = pickle.load(f)
        with open(f"{models_path}/combined_matrix.pkl", "rb") as f:
            self.combined_matrix = pickle.load(f)
        with open(f"{models_path}/product_profiles.pkl", "rb") as f:
            self.product_profiles = pickle.load(f)
        with open(f"{models_path}/product_idx_map.pkl", "rb") as f:
            maps = pickle.load(f)
            self.product_idx_map = maps["product_idx_map"]
            self.product_id_map = maps["product_id_map"]

    def recommend(self, user_id, n=10):
        user_review_count = len(self.df[self.df["user_id"] == user_id])
        if user_review_count < 5:
            als_w, cb_w = 0.2, 0.8
        elif user_review_count < 20:
            als_w, cb_w = 0.5, 0.5
        else:
            als_w, cb_w = 0.8, 0.2

        als_recs = self._als_recs(user_id, n=50)
        cb_recs = self._content_recs(user_id, n=50)

        all_products = set(als_recs.keys()) | set(cb_recs.keys())
        scores = {
            pid: als_w * als_recs.get(pid, 0) + cb_w * cb_recs.get(pid, 0)
            for pid in all_products
        }
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
        return [{"product_id": pid, "score": round(s, 4)} for pid, s in top]

    def _als_recs(self, user_id, n=20):
        if user_id not in self.user_encoder:
            return {}
        uid = self.user_encoder[user_id]
        ids, scores = self.als_model.recommend(
            uid, self.interaction_matrix.T[uid], N=n,
            filter_already_liked_items=True
        )
        mx = max(scores) if len(scores) > 0 else 1
        return {self.product_decoder[p]: float(s/mx) for p, s in zip(ids, scores)}

    def _content_recs(self, user_id, n=20):
        liked = self.df[(self.df["user_id"]==user_id) & (self.df["rating"]>=4)]
        if liked.empty:
            return {}
        acc = np.zeros(len(self.product_profiles))
        for _, row in liked.iterrows():
            if row["product_id"] not in self.product_idx_map:
                continue
            idx = self.product_idx_map[row["product_id"]]
            sims = cosine_similarity(
                self.combined_matrix[idx], self.combined_matrix
            ).flatten()
            acc += sims * row["rating"]
        rated = set(self.df[self.df["user_id"]==user_id]["product_id"])
        for pid in rated:
            if pid in self.product_idx_map:
                acc[self.product_idx_map[pid]] = 0
        mx = acc.max()
        if mx > 0:
            acc = acc / mx
        top = acc.argsort()[::-1][:n]
        return {self.product_id_map[i]: float(acc[i]) for i in top if acc[i] > 0}
'''

with open('../src/recommender.py', 'w') as f:
    f.write(hybrid_code)

print("✅ Hybrid recommender saved to src/recommender.py")
print("\nProject structure so far:")
for root, dirs, files in os.walk('..'):
    dirs[:] = [d for d in dirs if d not in ['.ipynb_checkpoints', '__pycache__']]
    level = root.replace('..', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

✅ Hybrid recommender saved to src/recommender.py

Project structure so far:
../
  requirements.txt
  api/
  data/
    clean_ratings.csv
    Musical_Instruments.json.gz
    rating_distribution.png
  models/
    als_model.pkl
    combined_matrix.pkl
    encoders.pkl
    interaction_matrix.pkl
    product_idx_map.pkl
    product_profiles.pkl
    tfidf_vectorizer.pkl
  notebooks/
    01_eda.ipynb
    02_collaborative_filtering.ipynb
    03_content_based.ipynb
    04_hybrid_model.ipynb
    05_evaluation.ipynb
  src/
    recommender.py
  ui/
